# MaaS validation demo (notebook)

Walkthrough: MaaS URL + OpenShift auth → create API key → list models → one completion → rate-limit loop.

**Prereq:** Python 3.9+ (stdlib: `urllib`, `http.client`, `getpass`).

**RHOAI:** **Demo quick swap** cell for `MAAS_BASE` + token between takes. `oc` runs on your laptop, not in the kernel — see README *OpenShift CLI and console*.

**Auth:** `OPENSHIFT_TOKEN` or username/password (§1). Filled **`DEMO_*`** overrides env.

## Demo quick swap

Sets demo-time overrides for gateway URL and OpenShift token. Run this cell, then run **§1 Configuration**.

| Variable | Effect |
|----------|--------|
| `DEMO_MAAS_BASE` | Non-empty → overrides `MAAS_BASE` env. |
| `DEMO_OPENSHIFT_TOKEN` | Non-empty → overrides `OPENSHIFT_TOKEN` env. |

In [ ]:
DEMO_MAAS_BASE = ""
DEMO_OPENSHIFT_TOKEN = ""

## 1. Configuration

Loads env (and optional OAuth password flow). Defines `MAAS_BASE`, `API_KEYS_URL`, `MODELS_URL`, resolves OpenShift token, prints non-secret summary.

| Variable | Required | Purpose |
|----------|----------|---------|
| `MAAS_BASE` | Yes | MaaS gateway origin (no trailing slash). |
| `OPENSHIFT_TOKEN` | token *or* login | For `POST /api-keys`. |
| `OPENSHIFT_USERNAME` / `OPENSHIFT_PASSWORD` | If login | OAuth basic-auth token fetch. |
| `OPENSHIFT_PASSWORD_FILE` | Optional | First line = password. |
| `OPENSHIFT_OAUTH_URL` | Optional | OAuth host; else derived from `MAAS_BASE` when `*.apps.*`. |
| `MAAS_SUBSCRIPTION` | No | Subscription name (default `simulator-subscription`). |
| `VERIFY_TLS` | No | `1` / `true` for strict TLS. |

In [ ]:
import base64
import getpass
import http.client
import json
import os
import ssl
import urllib.error
import urllib.request
from typing import Any, Dict, Optional
from urllib.parse import parse_qs, urlencode, urlparse

# --- 1a) Environment variables (set here or export before Jupyter) ---
# DEMO_MAAS_BASE / DEMO_OPENSHIFT_TOKEN from "Demo quick swap" cell override env when non-empty.

_maas_demo = globals().get("DEMO_MAAS_BASE", "")
if isinstance(_maas_demo, str) and _maas_demo.strip():
    MAAS_BASE = _maas_demo.strip()
else:
    MAAS_BASE = os.environ.get("MAAS_BASE", "https://maas.YOUR_DOMAIN_HERE")

_ot_demo = globals().get("DEMO_OPENSHIFT_TOKEN", "")
if isinstance(_ot_demo, str) and _ot_demo.strip():
    OPENSHIFT_TOKEN = _ot_demo.strip()
else:
    OPENSHIFT_TOKEN = os.environ.get("OPENSHIFT_TOKEN", "").strip()

# Optional: obtain token via OpenShift OAuth (basic auth) instead of pasting a token
OPENSHIFT_USERNAME = os.environ.get("OPENSHIFT_USERNAME", "").strip()

# Shared demo password: prefer export OPENSHIFT_PASSWORD='...' so it never appears in the notebook
OPENSHIFT_PASSWORD = os.environ.get("OPENSHIFT_PASSWORD", "").strip()
# Or first line of a file (path only in env — keep the file outside git)
OPENSHIFT_PASSWORD_FILE = os.environ.get("OPENSHIFT_PASSWORD_FILE", "").strip()

# If username is set, password is missing, and this is false, prompt with hidden input (works best in a terminal; in some Jupyter UIs you may prefer env/file only)
SKIP_PASSWORD_PROMPT = os.environ.get("SKIP_PASSWORD_PROMPT", "").lower() in ("1", "true", "yes")

# OAuth server (override if auto-detect fails). Example: https://oauth-openshift.apps.<cluster>
OPENSHIFT_OAUTH_URL = os.environ.get("OPENSHIFT_OAUTH_URL", "").strip()

# Optional: hard-code overrides here (leave commented to use only environment variables)
# MAAS_BASE = "https://maas.apps.example.com"
# OPENSHIFT_USERNAME = "demo"
# OPENSHIFT_PASSWORD = ""  # prefer export OPENSHIFT_PASSWORD for a shared demo password

SUBSCRIPTION_NAME = os.environ.get("MAAS_SUBSCRIPTION", "simulator-subscription")
API_KEY_NAME = "notebook-demo-key"
API_KEY_EXPIRES = "24h"

# VERIFY_TLS: default off (curl -k style). Set VERIFY_TLS=1 or true in the environment for strict certs.
VERIFY_TLS = os.environ.get("VERIFY_TLS", "").lower() in ("1", "true", "yes")


def _oauth_base_from_maas_base(maas_base: str) -> Optional[str]:
    """Derive https://oauth-openshift.apps.<suffix> from https://something.apps.<suffix>."""
    u = urlparse(maas_base)
    host = (u.hostname or "").lower()
    if ".apps." not in host:
        return None
    suffix = host.split(".apps.", 1)[1]
    return f"{u.scheme}://oauth-openshift.apps.{suffix}"


def openshift_token_from_password(
    oauth_base: str,
    username: str,
    password: str,
    *,
    verify_tls: bool,
) -> str:
    """
    GET /oauth/authorize?client_id=openshift-challenging-client&response_type=token
    with Basic auth + X-CSRF-Token; read access_token from the 302 Location URL fragment.
    """
    oauth_base = oauth_base.rstrip("/")
    auth_path = "/oauth/authorize?" + urlencode(
        {"client_id": "openshift-challenging-client", "response_type": "token"}
    )
    basic = base64.b64encode(f"{username}:{password}".encode("utf-8")).decode("ascii")
    u = urlparse(oauth_base)
    if u.scheme != "https":
        raise ValueError("OPENSHIFT_OAUTH_URL must use https://")
    if not u.hostname:
        raise ValueError("OPENSHIFT_OAUTH_URL is missing a hostname")

    ctx = ssl.create_default_context()
    if not verify_tls:
        ctx.check_hostname = False
        ctx.verify_mode = ssl.CERT_NONE

    conn = http.client.HTTPSConnection(u.hostname, port=u.port or 443, context=ctx, timeout=120)
    headers = {
        "Authorization": f"Basic {basic}",
        "X-CSRF-Token": "notebook",
        "Accept": "*/*",
        "User-Agent": "maas-validation-notebook/1.0",
    }
    conn.request("GET", auth_path, headers=headers)
    resp = conn.getresponse()
    resp.read()
    location = resp.getheader("Location")
    conn.close()

    if resp.status not in (302, 303, 307) or not location:
        raise RuntimeError(
            f"OAuth login failed (HTTP {resp.status}). "
            "If your IdP disallows basic auth, set OPENSHIFT_TOKEN (e.g. oc whoami -t) instead."
        )

    frag = urlparse(location).fragment
    if not frag:
        raise RuntimeError(f"OAuth redirect had no fragment in Location: {location!r}")
    token = parse_qs(frag).get("access_token", [None])[0]
    if not token:
        raise RuntimeError(f"No access_token in OAuth redirect fragment: {location!r}")
    return token


# --- 1b) Resolve OpenShift token (env token wins) ---

MAAS_BASE = MAAS_BASE.rstrip("/")
API_KEYS_URL = f"{MAAS_BASE}/maas-api/v1/api-keys"
MODELS_URL = f"{MAAS_BASE}/maas-api/v1/models"

_oauth_base = OPENSHIFT_OAUTH_URL or _oauth_base_from_maas_base(MAAS_BASE)

if OPENSHIFT_TOKEN:
    print("Using OPENSHIFT_TOKEN from the environment (username/password skipped).")
elif OPENSHIFT_USERNAME:
    _pw = OPENSHIFT_PASSWORD
    if not _pw and OPENSHIFT_PASSWORD_FILE:
        with open(OPENSHIFT_PASSWORD_FILE, encoding="utf-8") as _f:
            _pw = _f.readline().strip()
    if not _pw and not SKIP_PASSWORD_PROMPT:
        _pw = getpass.getpass("OpenShift password (hidden; not echoed): ")
    if not _pw:
        raise SystemExit(
            "Set OPENSHIFT_PASSWORD, OPENSHIFT_PASSWORD_FILE, or OPENSHIFT_TOKEN, "
            "or enable a prompt-capable environment."
        )
    if not _oauth_base:
        raise SystemExit(
            "Could not derive OAuth URL from MAAS_BASE. Set OPENSHIFT_OAUTH_URL to e.g. "
            "https://oauth-openshift.apps.<your-cluster-domain>"
        )
    OPENSHIFT_TOKEN = openshift_token_from_password(
        _oauth_base, OPENSHIFT_USERNAME, _pw, verify_tls=VERIFY_TLS
    )
    print("Fetched OPENSHIFT_TOKEN via username/password (OAuth).")
else:
    print("No OPENSHIFT_TOKEN or OPENSHIFT_USERNAME — set one of them before creating an API key.")

# --- Summary (no secrets printed) ---

print("MAAS_BASE      :", MAAS_BASE)
print("API_KEYS_URL   :", API_KEYS_URL)
print("MODELS_URL     :", MODELS_URL)
print("SUBSCRIPTION   :", SUBSCRIPTION_NAME)
print("VERIFY_TLS     :", VERIFY_TLS)
print("OAuth base     :", _oauth_base or "(not set)")
print("OpenShift user :", OPENSHIFT_USERNAME or "(not set)")
print("Token ready    :", bool(OPENSHIFT_TOKEN))

## 2. Create an API key

`POST /maas-api/v1/api-keys` with OpenShift bearer token. Response `key` stored as **`API_KEY`** (shown once).

In [ ]:
def http_json(
    method: str,
    url: str,
    *,
    token: Optional[str] = None,
    data: Optional[Dict[str, Any]] = None,
):
    """Minimal JSON HTTP helper (stdlib only)."""
    headers = {"Content-Type": "application/json", "Accept": "application/json"}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    body = None
    if data is not None:
        body = json.dumps(data).encode("utf-8")
    ctx = ssl.create_default_context()
    if not VERIFY_TLS:
        ctx.check_hostname = False
        ctx.verify_mode = ssl.CERT_NONE
    req = urllib.request.Request(url, data=body, headers=headers, method=method)
    try:
        with urllib.request.urlopen(req, context=ctx, timeout=120) as resp:
            raw = resp.read().decode("utf-8")
            return resp.status, json.loads(raw) if raw else {}
    except urllib.error.HTTPError as e:
        err_body = e.read().decode("utf-8", errors="replace")
        try:
            parsed = json.loads(err_body) if err_body else {}
        except json.JSONDecodeError:
            parsed = {"_raw": err_body}
        raise RuntimeError(f"HTTP {e.code}: {parsed}") from None


if not OPENSHIFT_TOKEN:
    raise SystemExit("Set OPENSHIFT_TOKEN in the setup cell (or environment) to create an API key.")

payload = {
    "name": API_KEY_NAME,
    "description": "MaaS notebook validation demo",
    "expiresIn": API_KEY_EXPIRES,
    "subscription": SUBSCRIPTION_NAME,
}

_, body = http_json("POST", API_KEYS_URL, token=OPENSHIFT_TOKEN, data=payload)
API_KEY = body.get("key")
if not API_KEY:
    raise RuntimeError(f"No 'key' in response: {body}")

print("API key created (prefix):", API_KEY[:20] + "…")

## 3. Model list (discovery)

`GET /maas-api/v1/models` with **`API_KEY`**. Parses first model → **`MODEL_NAME`**, **`MODEL_URL`**.

In [ ]:
_B = "\033[1m"
_R = "\033[0m"

_, models_body = http_json("GET", MODELS_URL, token=API_KEY)

data = models_body.get("data") or []
if not data:
    raise SystemExit("No models in response; deploy a model and check subscription binding.")

first = data[0]
MODEL_NAME = first.get("id") or first.get("name")
MODEL_URL = (first.get("url") or "").rstrip("/")
if not MODEL_NAME or not MODEL_URL:
    raise RuntimeError(f"Could not parse model id/url from: {first}")

print()
print("┌" + "─" * 58 + "┐")
print("│  " + _B + "MODEL_NAME" + _R + "  " + MODEL_NAME.ljust(42) + " │")
print("│  " + _B + "MODEL_URL " + _R + "  " + MODEL_URL[:42].ljust(42) + " │")
print("└" + "─" * 58 + "┘")
if len(MODEL_URL) > 42:
    print("   " + MODEL_URL[42:])
print()
print(_B + "Raw (truncated)" + _R)
print(json.dumps(models_body, indent=2)[:3500])

## 4. Single inference call

`POST {MODEL_URL}/v1/completions` with **`API_KEY`**. Prints usage tokens and completion text.

In [ ]:
_B = "\033[1m"
_R = "\033[0m"

COMPLETIONS_URL = f"{MODEL_URL}/v1/completions"
inference_payload = {
    "model": MODEL_NAME,
    "prompt": "Hello from the notebook demo.",
    "max_tokens": 50,
}

status, completion = http_json("POST", COMPLETIONS_URL, token=API_KEY, data=inference_payload)

usage = completion.get("usage") or {}
choices = completion.get("choices") or []
choice0 = choices[0] if choices and isinstance(choices[0], dict) else {}
completion_text = choice0.get("text") or ""

print()
print(_B + "HTTP" + _R, status)
print()
print("┌" + "─" * 58 + "┐")
print("│  " + _B + "Tokens (usage)" + _R + " " * 29 + "│")
print("│    prompt_tokens      " + str(usage.get("prompt_tokens", "—")).ljust(31) + " │")
print("│    completion_tokens  " + str(usage.get("completion_tokens", "—")).ljust(31) + " │")
print("│    total_tokens       " + str(usage.get("total_tokens", "—")).ljust(31) + " │")
print("└" + "─" * 58 + "┘")
print()
print(_B + "Completion text" + _R)
print(completion_text if completion_text else "(empty)")
print()
print(_B + "Full JSON (truncated)" + _R)
print(json.dumps(completion, indent=2)[:4000])

## 5. Rate-limit probe

Repeated `POST` completions. Optional delay between requests. Stops after **`CONSECUTIVE_429_TO_STOP`** consecutive **429** responses, or **`MAX_REQUESTS`** total.

In [ ]:
import time

# --- tune here ---
SLEEP_BETWEEN_REQUESTS_SEC = 1.0  # 0 to disable pause between requests
CONSECUTIVE_429_TO_STOP = 3
MAX_REQUESTS = 64

inference_payload = {
    "model": MODEL_NAME,
    "prompt": "Rate limit probe.",
    "max_tokens": 50,
}

ctx = ssl.create_default_context()
if not VERIFY_TLS:
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE

consecutive_429 = 0
last_code = None
for i in range(1, MAX_REQUESTS + 1):
    body = json.dumps(inference_payload).encode("utf-8")
    req = urllib.request.Request(
        COMPLETIONS_URL,
        data=body,
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json",
        },
        method="POST",
    )
    try:
        with urllib.request.urlopen(req, context=ctx, timeout=120) as resp:
            last_code = resp.status
    except urllib.error.HTTPError as e:
        last_code = e.code

    print(f"{i:3d}  HTTP {last_code}", end="")
    if last_code == 429:
        consecutive_429 += 1
        print(f"  (429 streak {consecutive_429}/{CONSECUTIVE_429_TO_STOP})")
        if consecutive_429 >= CONSECUTIVE_429_TO_STOP:
            print(f"Stopping: {CONSECUTIVE_429_TO_STOP} consecutive 429 responses.")
            break
    else:
        consecutive_429 = 0
        print()

    if SLEEP_BETWEEN_REQUESTS_SEC > 0 and i < MAX_REQUESTS:
        time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)
else:
    if consecutive_429 < CONSECUTIVE_429_TO_STOP:
        print(f"Stopped: reached MAX_REQUESTS={MAX_REQUESTS} without {CONSECUTIVE_429_TO_STOP} consecutive 429s.")